In [4]:
from collections import defaultdict
import csv
from fitting_GUI import *

In [5]:
def load_table(path):
    table = defaultdict(list) #element -> [(line, energy, intensity), ...]
    with open(path, newline="") as fh:
        for r in csv.DictReader(fh):
            table[r["element"]].append((r["line"], float(r["energy"]), float(r["intensity"])))
    return table

def weighted_energy(lines, prefix):
    """Intensity-weighted mean energy of all lines starting with 'Ka' or 'Kb'."""
    pts = [(E, I) for name, E, I in lines if name.startswith(prefix)]
    return sum(E * I for E, I in pts) / sum(I for _, I in pts)

def identify(r, table, top=3):
    """r = results[folder][key]. Compares the two fitted peaks with each element."""
    peaks = [r["peak1"], r["peak2"]]
    if None in peaks:
        print("Need two successful fits")
        return
    a, b = sorted(peaks, key=lambda p: p["mu"]) #lower energy = Kα, higher = Kβ
    sig_a, sig_b = a["fwhm"] / 2.3548, b["fwhm"] / 2.3548

    scores = []
    for el, lines in table.items():
        Ea, Eb = weighted_energy(lines, "Ka"), weighted_energy(lines, "Kb")
        chi2 = ((a["mu"] - Ea) / sig_a) ** 2 + ((b["mu"] - Eb) / sig_b) ** 2
        scores.append((chi2, el, Ea, Eb))

    print(f"measured: Kα {a['mu']:.2f} keV, Kβ {b['mu']:.2f} keV")
    for chi2, el, Ea, Eb in sorted(scores)[:top]:
        print(f"  {el:>3}: Kα {Ea:.2f}, Kβ {Eb:.2f}  (score {chi2:.2f})")

In [6]:
results = load_results()

table = load_table("xray_energies.csv")

for folder, d in results.items():
    if folder == "calibration":
        continue
    for key, r in d.items():
        print(f"\n{folder}/{key}")
        identify(r, table)


measurements/week2_metal_1
Need two successful fits

measurements/week2_metal_2
Need two successful fits

measurements/week2_metal_3
measured: Kα 490.26 keV, Kβ 562.66 keV
   Po: Kα 78.37, Kβ 90.16  (score 50.93)
   Bi: Kα 76.25, Kβ 87.69  (score 51.47)
   Pb: Kα 74.16, Kβ 85.27  (score 51.99)

measurements/week3_metal_1
measured: Kα 402.36 keV, Kβ 456.66 keV
   Po: Kα 78.37, Kβ 90.16  (score 502.83)
   Bi: Kα 76.25, Kβ 87.69  (score 509.53)
   Pb: Kα 74.16, Kβ 85.27  (score 516.14)

measurements/week3_metal_2
measured: Kα 465.03 keV, Kβ 533.22 keV
   Po: Kα 78.37, Kβ 90.16  (score 731.97)
   Bi: Kα 76.25, Kβ 87.69  (score 740.09)
   Pb: Kα 74.16, Kβ 85.27  (score 748.11)

measurements/week3_metal_3
measured: Kα 507.01 keV, Kβ 579.81 keV
   Po: Kα 78.37, Kβ 90.16  (score 685.31)
   Bi: Kα 76.25, Kβ 87.69  (score 692.17)
   Pb: Kα 74.16, Kβ 85.27  (score 698.93)
